In [19]:
import os
import pandas as pd
import numpy as np 
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

In [2]:
# Se cargan los datos. 

data=pd.read_csv('car_details_v3.csv',sep=',')

In [3]:
# Cantidad de datos y número de variables

data.shape

(8128, 13)

In [4]:
# Mostrar los datos

data.head()

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats
0,Maruti Swift Dzire VDI,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.4 kmpl,1248 CC,74 bhp,190Nm@ 2000rpm,5.0
1,Skoda Rapid 1.5 TDI Ambition,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14 kmpl,1498 CC,103.52 bhp,250Nm@ 1500-2500rpm,5.0
2,Honda City 2017-2020 EXi,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.7 kmpl,1497 CC,78 bhp,"12.7@ 2,700(kgm@ rpm)",5.0
3,Hyundai i20 Sportz Diesel,2010,225000,127000,Diesel,Individual,Manual,First Owner,23.0 kmpl,1396 CC,90 bhp,22.4 kgm at 1750-2750rpm,5.0
4,Maruti Swift VXI BSIII,2007,130000,120000,Petrol,Individual,Manual,First Owner,16.1 kmpl,1298 CC,88.2 bhp,"11.5@ 4,500(kgm@ rpm)",5.0


In [5]:
# Es recomendable que todos los pasos preparación se realicen sobre otro archivo.

data_t = data

In [6]:
data_t=data_t.dropna()
data_t=data_t.drop_duplicates()

data_t['mileage'] = data_t['mileage'].astype(str).str.extract(r'([\d.]+)').astype(float)
data_t['mileage'] = pd.to_numeric(data_t['mileage'], errors='coerce')
data_t['engine'] = data_t['engine'].astype(str).str.extract(r'([\d.]+)').astype(float)
data_t['engine'] = pd.to_numeric(data_t['engine'], errors='coerce')
data_t['max_power'] = data_t['max_power'].astype(str).str.extract(r'([\d.]+)').astype(float)
data_t['max_power'] = pd.to_numeric(data_t['max_power'], errors='coerce')

data_t.shape

(6717, 13)

In [7]:
# Transformación de los datos

data_t = pd.get_dummies(data_t, columns=['fuel','owner'],dtype=int)

In [8]:
# Eliminación de las variables que no son numericas.


data_t=data_t.drop(['name'], axis=1)
data_t=data_t.drop(['seller_type'], axis=1)
data_t=data_t.drop(['transmission'], axis=1)
data_t=data_t.drop(['torque'], axis=1)

In [9]:
data_t.shape

(6717, 16)

In [10]:
data_t.head(5)

,year,selling_price,km_driven,mileage,engine,max_power,seats,fuel_CNG,fuel_Diesel,fuel_LPG,fuel_Petrol,owner_First Owner,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,2014,450000,145500,23.40,1248.0,74.00,5.0,0,1,0,0,1,0,0,0,0
1,2014,370000,120000,21.14,1498.0,103.52,5.0,0,1,0,0,0,0,1,0,0
2,2006,158000,140000,17.70,1497.0,78.00,5.0,0,0,0,1,0,0,0,0,1
3,2010,225000,127000,23.00,1396.0,90.00,5.0,0,1,0,0,1,0,0,0,0
4,2007,130000,120000,16.10,1298.0,88.20,5.0,0,0,0,1,1,0,0,0,0


In [11]:
# Se selecciona la variable objetivo, en este caso "precio".

Y=data_t['selling_price']
X=data_t.drop(['selling_price'], axis=1)

In [12]:
# Utilizaremos una tranformación de grado 2.

poly = PolynomialFeatures(degree=2)
poly_X = poly.fit_transform(X)

In [13]:
# Esta transformación crea nuevas variables y las añade al conjunto de datos. Veamos cuántas se generan:

poly_X.shape

(6717, 136)

In [14]:
# Se realiza la división entrenamiento - test. Se deja 20% de los datos para el test.

poly_X_train, poly_X_test, poly_Y_train, poly_Y_test = train_test_split(poly_X, Y, test_size = 0.2, random_state = 0)

In [15]:
# Creación del objeto de la clase LinearRegression y ajuste del modelo a los datos.

modelo_regresion_poly = LinearRegression()
modelo_regresion_poly

LinearRegression()

In [16]:
# Ajustar el modelo con los datos de entrenamiento con las nuevas variables polinomiales

modelo_regresion_poly.fit(poly_X_train, poly_Y_train)

LinearRegression()

In [17]:
# Se obtienen las predicciones del modelo sobre el conjunto test.

y_pred = modelo_regresion_poly.predict(poly_X_test)

In [20]:
# Se obtienen las métricas a partir de la predicción y la base de evaluación (valores reales).

mse = mean_squared_error(poly_Y_test, y_pred)
rmse = np.sqrt(mse)   
mae = mean_absolute_error(poly_Y_test, y_pred)
r2 = r2_score(poly_Y_test, y_pred)

print("MSE: %.2f" % mse)
print("RMSE: %.2f" % rmse)
print("MAE: %.2f" % mae)
print("R²: %.2f" % r2)

MSE: 108183859448.39
RMSE: 328913.15
MAE: 131746.80
R²: 0.66


In [21]:
# Se realiza la división entrenamiento - test. Como estamos utilizando el mismo valor para random_state (=0) 
# garantizamos que obtenemos la misma partición utilizada para el modelo de regresión polinomial.

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=0)
modelo_reg_lineal = LinearRegression().fit(X_train, Y_train)

y_pred = modelo_reg_lineal.predict(X_test)

mse = mean_squared_error(poly_Y_test, y_pred)
rmse = np.sqrt(mse)   
mae = mean_absolute_error(poly_Y_test, y_pred)
r2 = r2_score(poly_Y_test, y_pred)

print("MSE: %.2f" % mse)
print("RMSE: %.2f" % rmse)
print("MAE: %.2f" % mae)
print("R²: %.2f" % r2)

MSE: 129372479578.93
RMSE: 359683.86
MAE: 181378.90
R²: 0.60
